# Diemtigen – Interpretable Inputs (CSV generation) 
This notebook generates: - `data_features.csv` (phase-based split features on 3 segments)
- `data_lowresspec.csv` (low-resolution spectrogram features on post-P window) **Dataset inputs (already available locally):**
- `events_mainshocks_foreshocks_aftershocks_15sec_23days.h5`
- `info_h5_events_mainshocks_foreshocks_aftershocks_15sec_23days.csv` Assumptions (verified from HDF5 attrs):
- 3 components: **HHE, HHN, HHZ**
- sampling rate: **120 Hz**
- duration: **15 s** (~1800 samples; stored as 1801 points) Segmentation (P fixed at 5 s):
- `[0,5)` noise
- `[5,8)` P + early
- `[8,15]` coda

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py

data_root = Path("diemtigen_data")

h5_path = data_root / "events_mainshocks_foreshocks_aftershocks_15sec_23days.h5"
info_path = data_root / "info_h5_events_mainshocks_foreshocks_aftershocks_15sec_23days.csv"

info = pd.read_csv(info_path)
info.head()

,id,detection_time,CC,amplitude,template_id,detection_type,detected_template_id,config_tm,preferred_magnitude,mag_type,...,location_type,origintime,latitude,longitude,depth,quakeml_file,stream_file,event_type,time_diff,event_id
0,511,2014-04-17 01:51:56,0.64,737.299988,2,D,NaN,2,1,MLh,...,NaN,NaN,46.654654,7.586811,8076.171875,NaN,NaN,foreshock,2531.0,20140417015156
1,512,2014-04-17 02:34:07,0.76,88.800003,2,D,NaN,2,1,MLh,...,NaN,NaN,46.654654,7.586811,8076.171875,NaN,NaN,foreshock,1690.0,20140417023407
2,513,2014-04-17 03:02:17,0.50,1371.599976,1,D,NaN,2,1,MLh,...,NaN,NaN,46.654654,7.590898,8431.640625,NaN,NaN,foreshock,201.0,20140417030217
3,515,2014-04-17 03:05:38,0.74,280.500000,1,D,122.0,2,1,MLh,...,NaN,NaN,46.655471,7.590877,8380.859375,NaN,NaN,foreshock,60.0,20140417030538
4,516,2014-04-17 03:06:38,0.48,27.799999,1,D,NaN,2,1,MLh,...,NaN,NaN,46.655471,7.590877,8380.859375,NaN,NaN,foreshock,808.0,20140417030638


In [2]:
# Keep only foreshock/aftershock labels
label_map = {"foreshock": 0, "aftershock": 1}
info["label"] = info["event_type"].map(label_map)
info_fa = info[info["event_type"].isin(label_map.keys())].copy().reset_index(drop=True)

info_fa["event_type"].value_counts(), info_fa.shape

(event_type
 aftershock    1172
 foreshock     1169
 Name: count, dtype: int64,
 (2341, 23))

In [3]:
import math

def rms(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(np.sqrt(np.mean(x*x))) if x.size else float("nan")

def energy(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(np.sum(x*x)) if x.size else float("nan")

def peak_abs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(np.max(np.abs(x))) if x.size else float("nan")

def std(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(np.std(x)) if x.size else float("nan")

def crest_factor(x: np.ndarray) -> float:
    r = rms(x)
    p = peak_abs(x)
    return float(p / r) if (r and not np.isnan(r) and r > 0) else float("nan")

def zero_crossing_rate(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if x.size < 2:
        return float("nan")
    s = np.sign(x)
    s[s == 0] = 1
    return float(np.mean(s[:-1] != s[1:]))

def envelope_hilbert(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    from scipy.signal import hilbert
    return np.abs(hilbert(x))

def env_max(x: np.ndarray) -> float:
    e = envelope_hilbert(x)
    return float(np.max(e)) if e.size else float("nan")

def env_area(x: np.ndarray) -> float:
    e = envelope_hilbert(x)
    return float(np.sum(e)) if e.size else float("nan")

def env_decay_slope(x: np.ndarray, sr: float) -> float:
    e = envelope_hilbert(x)
    if e.size < 10:
        return float("nan")
    e = np.maximum(e, 1e-12)
    t = np.arange(e.size) / float(sr)
    y = np.log(e)
    k0 = int(0.05 * e.size)
    t2, y2 = t[k0:], y[k0:]
    if t2.size < 10:
        return float("nan")
    A = np.vstack([t2, np.ones_like(t2)]).T
    slope, _ = np.linalg.lstsq(A, y2, rcond=None)[0]
    return float(slope)

def psd_features(x: np.ndarray, sr: float, bands=((1,5),(5,10),(10,20),(20,45))):
    from scipy.signal import welch
    x = np.asarray(x, dtype=float)
    out = {}
    if x.size < 32:
        for (a,b) in bands:
            out[f"band_{a}_{b}_pow"] = float("nan")
        out["spec_centroid"] = float("nan")
        out["spec_bandwidth"] = float("nan")
        return out
    f, Pxx = welch(x, fs=sr, nperseg=min(256, x.size))
    for (a,b) in bands:
        m = (f >= a) & (f < b)
        out[f"band_{a}_{b}_pow"] = float(np.trapezoid(Pxx[m], f[m])) if np.any(m) else 0.0
    P = np.maximum(Pxx, 0)
    denom = np.sum(P)
    if denom <= 0:
        out["spec_centroid"] = float("nan")
        out["spec_bandwidth"] = float("nan")
    else:
        centroid = float(np.sum(f * P) / denom)
        bandwidth = float(np.sqrt(np.sum(((f - centroid)**2) * P) / denom))
        out["spec_centroid"] = centroid
        out["spec_bandwidth"] = bandwidth
    return out

def lowres_spec_features(x: np.ndarray, sr: float, t_start: float, t_end: float, bin_hz: float=10.0):
    x = np.asarray(x, dtype=float)
    i0 = int(round(t_start * sr))
    i1 = int(round(t_end * sr))
    seg = x[i0:i1]
    nsec = int(round(t_end - t_start))
    target = int(round(nsec * sr))
    if seg.size < target:
        seg = np.pad(seg, (0, target - seg.size))
    elif seg.size > target:
        seg = seg[:target]
    seg = seg.reshape(nsec, int(round(sr)))
    nyq = sr / 2.0
    edges = np.arange(0.0, nyq + 1e-9, bin_hz)
    if edges[-1] < nyq:
        edges = np.append(edges, nyq)
    out = {}
    for ti in range(nsec):
        w = seg[ti] * np.hanning(seg.shape[1])
        X = np.fft.rfft(w)
        freqs = np.fft.rfftfreq(w.size, d=1/sr)
        P = (np.abs(X)**2) / w.size
        for bi in range(len(edges)-1):
            lo, hi = edges[bi], edges[bi+1]
            m = (freqs >= lo) & (freqs < hi) if bi < len(edges)-2 else (freqs >= lo) & (freqs <= hi)
            out[f"spec_t{ti}_f{int(lo)}_{int(hi)}"] = float(np.sum(P[m])) if np.any(m) else 0.0
    return out

In [4]:
# Segment definitions (seconds) and channel mapping
SEGMENTS = {
    "noise": (0.0, 5.0),
    "p": (5.0, 8.0),
    "coda": (8.0, 15.0),
}

CHANNELS = {
    "E": "HHE",
    "N": "HHN",
    "Z": "HHZ",
}

def get_event_traces(h5f, event_id: str):
    g = h5f[str(event_id)]
    tr = g["traces"]
    keys = list(tr.keys())
    sr = float(tr[keys[0]].attrs.get("sampling_rate", np.nan))
    out = {}
    for comp, chan in CHANNELS.items():
        k = [kk for kk in keys if kk.endswith(chan)]
        out[comp] = np.array(tr[k[0]][:]) if k else None
    return out, sr, keys

# Quick sanity check on one event
with h5py.File(h5_path, "r") as h5f:
    e0 = str(info_fa.loc[0, "event_id"])
    traces, sr, k = get_event_traces(h5f, e0)
    print("event:", e0, "sr:", sr, "keys:", k)
    print({c: traces[c].shape for c in traces})

event: 20140417015156 sr: 120.0 keys: ['CH.WIMIS..HHE', 'CH.WIMIS..HHN', 'CH.WIMIS..HHZ']
{'E': (1801,), 'N': (1801,), 'Z': (1801,)}


In [5]:
# Generate data_features.csv and data_lowresspec.csv
rows_feat = []
rows_spec = []

with h5py.File(h5_path, "r") as h5f:
    for _, r in info_fa.iterrows():
        eid = str(r["event_id"])
        if eid not in h5f:
            continue

        traces, sr, trace_keys = get_event_traces(h5f, eid)
        if any(traces[c] is None for c in CHANNELS.keys()):
            continue

        # match lengths
        L = min(traces["E"].size, traces["N"].size, traces["Z"].size)
        for c in traces:
            traces[c] = traces[c][:L]

        H = np.sqrt(traces["E"].astype(float)**2 + traces["N"].astype(float)**2)

        base = {
            "event_id": eid,
            "label": int(r["label"]),
            "event_type": r["event_type"],
            "sampling_rate": sr,
            "npts": L,
        }

        # --- split features
        feat = dict(base)
        for seg_name, (t0, t1) in SEGMENTS.items():
            i0 = int(round(t0 * sr))
            i1 = int(round(t1 * sr))
            for comp in ["E", "N", "Z"]:
                x = traces[comp][i0:i1].astype(float)
                prefix = f"{comp}_{seg_name}"
                feat[f"{prefix}_rms"] = rms(x)
                feat[f"{prefix}_peak"] = peak_abs(x)
                feat[f"{prefix}_energy"] = energy(x)
                feat[f"{prefix}_std"] = std(x)
                feat[f"{prefix}_crest"] = crest_factor(x)
                feat[f"{prefix}_zcr"] = zero_crossing_rate(x)
                feat[f"{prefix}_envmax"] = env_max(x)
                feat[f"{prefix}_envarea"] = env_area(x)
                if seg_name == "coda":
                    feat[f"{prefix}_envdecay"] = env_decay_slope(x, sr)

                psd = psd_features(x, sr)
                for kk, vv in psd.items():
                    feat[f"{prefix}_{kk}"] = vv

        # ratios
        for comp in ["E","N","Z"]:
            rp = feat[f"{comp}_p_rms"]
            rn = feat[f"{comp}_noise_rms"]
            feat[f"{comp}_snr_rms_p_over_noise"] = (rp / rn) if (rn and not np.isnan(rn) and rn > 0) else np.nan

            ep = feat[f"{comp}_p_energy"]
            ec = feat[f"{comp}_coda_energy"]
            feat[f"{comp}_coda_over_p_energy"] = (ec / ep) if (ep and not np.isnan(ep) and ep > 0) else np.nan

        for seg_name in SEGMENTS.keys():
            zE = feat[f"Z_{seg_name}_energy"]
            t0, t1 = SEGMENTS[seg_name]
            i0 = int(round(t0 * sr))
            i1 = int(round(t1 * sr))
            hE = energy(H[i0:i1])
            feat[f"ZHratio_{seg_name}_energy"] = (zE / hE) if (hE and not np.isnan(hE) and hE > 0) else np.nan

        rows_feat.append(feat)

        # --- low-res spectrogram (post-P: 5–15s, 1s bins, 10Hz bands up to Nyquist)
        spec = dict(base)
        for comp in ["E","N","Z"]:
            x = traces[comp].astype(float)
            d = lowres_spec_features(x, sr=sr, t_start=5.0, t_end=15.0, bin_hz=10.0)
            for kk, vv in d.items():
                spec[f"{comp}_{kk}"] = vv
        rows_spec.append(spec)

df_features = pd.DataFrame(rows_feat)
df_spec = pd.DataFrame(rows_spec)

df_features.to_csv(data_root / "data_features.csv", index=False)
df_spec.to_csv(data_root / "data_lowresspec.csv", index=False)

df_features.shape, df_spec.shape

((2341, 143), (2341, 185))

In [6]:
# Quick peek
df_features.head()

,event_id,label,event_type,sampling_rate,npts,E_noise_rms,E_noise_peak,E_noise_energy,E_noise_std,E_noise_crest,...,Z_coda_spec_bandwidth,E_snr_rms_p_over_noise,E_coda_over_p_energy,N_snr_rms_p_over_noise,N_coda_over_p_energy,Z_snr_rms_p_over_noise,Z_coda_over_p_energy,ZHratio_noise_energy,ZHratio_p_energy,ZHratio_coda_energy
0,20140417015156,0,foreshock,120.0,1801,53.375790,120.0,1709385.0,48.988654,2.248210,...,6.760290,1.084079,2.431962,1.119927,5.549150,0.614863,4.094174,1.269386,0.400136,0.478053
1,20140417023407,0,foreshock,120.0,1801,90.630495,222.0,4928332.0,90.310153,2.449507,...,6.473917,1.147033,0.457310,1.343952,1.949502,0.795455,1.156741,1.453816,0.611481,0.616940
2,20140417030217,0,foreshock,120.0,1801,86.846435,186.0,4525382.0,83.307213,2.141711,...,6.078136,0.455075,2.429443,1.519907,1.466519,1.570888,1.292401,0.318000,1.395026,1.023748
3,20140417030538,0,foreshock,120.0,1801,74.423350,169.0,3323301.0,74.376215,2.270793,...,11.618606,1.290553,1.222610,0.896900,3.924260,0.810456,1.460216,0.989493,0.415860,0.437376
4,20140417030638,0,foreshock,120.0,1801,43.856185,117.0,1154019.0,40.634591,2.667811,...,4.397816,1.726916,1.882084,0.459145,2.846957,1.199102,3.299376,0.639783,0.924079,1.502978


In [7]:
df_spec.head()

,event_id,label,event_type,sampling_rate,npts,E_spec_t0_f0_10,E_spec_t0_f10_20,E_spec_t0_f20_30,E_spec_t0_f30_40,E_spec_t0_f40_50,...,Z_spec_t8_f20_30,Z_spec_t8_f30_40,Z_spec_t8_f40_50,Z_spec_t8_f50_60,Z_spec_t9_f0_10,Z_spec_t9_f10_20,Z_spec_t9_f20_30,Z_spec_t9_f30_40,Z_spec_t9_f40_50,Z_spec_t9_f50_60
0,20140417015156,0,foreshock,120.0,1801,26120.470740,59.688356,37.614742,10.424042,3235.464998,...,14.054786,8.392216,75.441422,357.040668,234903.046817,77.104478,43.851351,14.209113,75.847605,339.643414
1,20140417023407,0,foreshock,120.0,1801,30264.688373,443.162371,16.910206,22.708233,3494.158907,...,11.986653,17.480699,81.640753,336.805145,115271.181087,48.901452,22.195765,9.068187,66.952310,358.808979
2,20140417030217,0,foreshock,120.0,1801,30206.455578,46.262374,23.809516,27.532479,2853.598303,...,14.536349,5.997800,76.756209,363.898514,233339.883498,134.603870,29.275060,4.295784,62.796343,315.112716
3,20140417030538,0,foreshock,120.0,1801,39849.163437,119.943113,23.911850,21.842128,2458.393538,...,32.987273,2.683972,59.645354,342.864680,332978.160077,84.766430,67.575060,9.155019,86.616641,363.366526
4,20140417030638,0,foreshock,120.0,1801,13030.949452,101.292810,18.285475,14.706430,2475.038880,...,16.006717,4.969292,72.532353,296.760272,11056.885373,185.903414,29.398471,6.910128,58.177126,284.138988
